<div style="text-align: center; font-weight: bold;">
Narrative Based Severity Classification Model</div>
This project builds a model that reads medical‑device incident narratives and assigns a severity level to each one. The goal is to turn unstructured text, often the earliest source of information about device malfunctions or patient harm, into a clear, consistent severity signal that can be analyzed at scale. By classifying these narratives, we can better understand how devices fail, how often those failures lead to harm, and where emerging risks may be developing. This gives analysts and decision‑makers a faster, more reliable way to monitor safety trends and identify issues that require attention.

In [311]:
from sqlalchemy import create_engine
import pandas as pd
import numpy as np
import re
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from scipy.stats import chi2_contingency, f_oneway
from sklearn.impute import SimpleImputer

In [ ]:
# Database connection

engine = create_engine("postgresql+psycopg2://<username>:<password>@<host>:<port>/<database>")


Step 1: Pull the data (includes structured labels + text)

This ensures device_class and Add manufacturer name. Some manufacturers (Dexcom, Medtronic) dominate reports; this could help distinguish reporting behavior from actual risk.

In [313]:
query = """
    SELECT 
    n.report_number,
    n.text,
    e.event_type,
    e.adverse_event_flag_clean,
    e.product_problem_flag_clean,
    e.reporter_state_code,          -- Candidate Feature
    e.manufacturer_name,
    e.adverse_event_flag,
    p.outcome,
    p.patient_sex,                  -- Candidate Feature
    p.patient_age_clean,            
    d.device_operator,              -- Candidate Feature
    d.device_class,
    d.brand_name,
    d.manufacturer_d_name,
    d.generic_name,
    d.device_availability,          -- Candidate Feature
    d.medical_specialty             -- Candidate Feature
FROM medical_device.narratives_clean n
LEFT JOIN medical_device.events_clean e 
    ON n.report_number = e.report_number
LEFT JOIN (
    SELECT DISTINCT ON (report_number) 
        report_number, 
        outcome,
        patient_sex,
        patient_age_clean           
    FROM medical_device.patients_clean
    ORDER BY report_number, patient_sequence_number  -- Takes the first patient listed
) p ON n.report_number = p.report_number
LEFT JOIN (
    SELECT DISTINCT ON (report_number) 
        report_number, 
        device_class, 
        manufacturer_d_name,
        generic_name,
        device_operator,
        device_availability,
        medical_specialty,
        brand_name
    FROM medical_device.devices_clean
    ORDER BY report_number, device_event_key
) d ON n.report_number = d.report_number
WHERE n.text IS NOT NULL AND n.text <> ''
"""

In [314]:
df = pd.read_sql(query, engine)

print(f"Total narrative records: {len(df):,}")

Total narrative records: 1,151,992


In [315]:
# Manufacturer distribution

print("Manufacturer distribution (top 10):")
print(df['manufacturer_d_name'].value_counts().head(10))
print(f"\nNumber of unique manufacturers: {df['manufacturer_d_name'].nunique():,}")

Manufacturer distribution (top 10):
manufacturer_d_name
INSTITUT STRAUMANN AG                   485907
JJGC S.A.                               122450
DEXCOM, INC.                            105973
NOBEL BIOCARE AB GÖTEBORG                39708
MEDTRONIC PUERTO RICO OPERATIONS CO.     37203
TANDEM DIABETES CARE                     34410
ANTHOGYR                                 27912
MEDTRONIC MINIMED                        15335
INSULET CORPORATION                      12223
ABBOTT DIABETES CARE INC                 11904
Name: count, dtype: int64

Number of unique manufacturers: 1,889


In [316]:
print("Device class distribution:")
print(df['device_class'].value_counts(dropna=False))

Device class distribution:
device_class
2       1005777
3        120780
1         23717
U          1131
N           302
f           282
None          3
Name: count, dtype: int64


Limit each device class: This made the dataset more balanced across device classes.

In [317]:
# Limit each device class to a maximum of 30,000 rows
SAMPLE_SIZE_PER_CLASS = 25000

def limit_sample_per_class(df, col, n):
    """
    For each unique value in `col`, sample up to `n` rows.
    If a class has fewer than `n` rows, take all.
    """
    return df.groupby(col, group_keys=False).apply(
        lambda x: x.sample(min(len(x), n), random_state=42)
    )

# Apply the sampling
df_sampled = limit_sample_per_class(df, 'device_class', SAMPLE_SIZE_PER_CLASS)

# Check the new distribution
print("\nSampled device class distribution (max 30,000 per class):")
print(df_sampled['device_class'].value_counts(dropna=False))
print(f"\nNew total rows: {len(df_sampled):,}")


Sampled device class distribution (max 30,000 per class):
device_class
2       25000
3       25000
1       23717
U        1131
N         302
f         282
None        3
Name: count, dtype: int64

New total rows: 75,435


C:\Users\patro\AppData\Local\Temp\ipykernel_7920\4158456004.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(col, group_keys=False).apply(


In [318]:
# New variable to keep the original untouched, then replace for analysis
df = limit_sample_per_class(df, 'device_class', SAMPLE_SIZE_PER_CLASS)

C:\Users\patro\AppData\Local\Temp\ipykernel_7920\4158456004.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(col, group_keys=False).apply(


Modeling approach

Step 2: Create labels from STRUCTURED columns ONLY

In [319]:
def label_severity(row):
    event = str(row['event_type']).lower()
    outcome = str(row['outcome']).lower()
    
    # CRITICAL: Death or Life Threatening
    if event == 'death' or 'death' in outcome or 'life threatening' in outcome:
        return 'critical'
    
    # HIGH: Injury, Hospitalization, Required Intervention, Disability
    if event == 'injury' or 'hospital' in outcome or 'intervention' in outcome or 'disability' in outcome:
        return 'high'
    
    # MEDIUM (now includes Malfunction, Product Problems, Other, no harm)
    # Everything else that is not critical or high
    return 'medium'

In [320]:
df['severity'] = df.apply(label_severity, axis=1) 

In [321]:
# Encode Labels and Check Distribution
le = LabelEncoder()
df['severity_label'] = le.fit_transform(df['severity'])

print("Class distribution:")
print(df['severity'].value_counts())

Class distribution:
severity
medium      45042
high        29515
critical      878
Name: count, dtype: int64


Measure the association between each candidate feature and the target (severity_label)

In [322]:
# --- Helper: Cramér's V (for Categorical vs Categorical) ---
def cramers_v(contingency_table):
    """Calculate Cramér's V for a contingency table."""
    chi2 = chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    dof = min(contingency_table.shape[0], contingency_table.shape[1]) - 1
    if dof == 0 or n == 0:
        return 0.0
    return np.sqrt(chi2 / (n * dof))

In [323]:
# --- Helper: ANOVA Summary (for Numerical vs Categorical) ---
def anova_summary(df, numerical_col, target_col='severity_label'):
    """Print mean target per group and F-statistic."""
    groups = [group[numerical_col].dropna().values for name, group in df.groupby(target_col)]
    if len(groups) < 2:
        print("Not enough groups.")
        return
    f_stat, p_val = f_oneway(*groups)
    means = df.groupby(target_col)[numerical_col].mean()
    print(f"ANOVA for {numerical_col}:")
    print(f"  Mean Age per Severity:")
    for label, mean in means.items():
        print(f"    {label}: {mean:.2f}")
    print(f"  F-statistic: {f_stat:.2f}")
    print(f"  P-value: {p_val:.6f} (Ignore this - effect size matters more)")
    print("  Interpretation: Strong if means differ by > 5-10 years.")

In [324]:
# 1. RUN TESTS ON CATEGORICAL FEATURES (Cramér's V)

categorical_features = [
    'device_operator',
    'reporter_state_code', 
    'medical_specialty',
    'patient_sex',
    'device_availability',
    'event_type',
    'manufacturer_name',
    'adverse_event_flag',
    'brand_name',
    'device_operator'
]

print("===== CATEGORICAL FEATURE STRENGTH (Cramér's V) =====")
print("(Target: severity_label)\n")

for col in categorical_features:
    if col not in df.columns:
        print(f"Skipping {col} (not in DataFrame)")
        continue
    
    # Build contingency table (drop rows where feature is missing)
    table = pd.crosstab(df[col], df['severity_label'])
    
    # If table is too sparse, skip (or combine)
    if table.shape[0] < 2:
        print(f"{col}: Only 1 category -> Skipping")
        continue
    
    v = cramers_v(table)
    
    # Determine strength
    if v >= 0.15:
        strength = "✅ STRONG - Keep"
    elif v >= 0.08:
        strength = "🟡 MODERATE - Test"
    else:
        strength = "🔴 WEAK - Drop"
    
    print(f"{col}: Cramér's V = {v:.4f} -> {strength}")

===== CATEGORICAL FEATURE STRENGTH (Cramér's V) =====
(Target: severity_label)

device_operator: Cramér's V = 0.3742 -> ✅ STRONG - Keep
reporter_state_code: Cramér's V = 0.1714 -> ✅ STRONG - Keep
medical_specialty: Cramér's V = 0.5310 -> ✅ STRONG - Keep
patient_sex: Cramér's V = 0.4411 -> ✅ STRONG - Keep
device_availability: Cramér's V = 0.3058 -> ✅ STRONG - Keep
event_type: Cramér's V = 0.8591 -> ✅ STRONG - Keep
manufacturer_name: Only 1 category -> Skipping
adverse_event_flag: Cramér's V = 0.9636 -> ✅ STRONG - Keep
brand_name: Cramér's V = 0.7739 -> ✅ STRONG - Keep
device_operator: Cramér's V = 0.3742 -> ✅ STRONG - Keep


In [325]:
# 2. RUN TESTS ON NUMERICAL FEATURES (Age - ANOVA)

print("\n===== NUMERICAL FEATURE STRENGTH (ANOVA) =====")

# Ensure Age is numeric
df['patient_age_clean'] = pd.to_numeric(df['patient_age_clean'], errors='coerce')

if 'patient_age_clean' in df.columns and df['patient_age_clean'].notna().sum() > 100:
    anova_summary(df, 'patient_age_clean')
else:
    print("Skipping Age - not available or too many missing values.")


===== NUMERICAL FEATURE STRENGTH (ANOVA) =====
ANOVA for patient_age_clean:
  Mean Age per Severity:
    0: 66.09
    1: 58.11
    2: 53.46
  F-statistic: 335.93
  P-value: 0.000000 (Ignore this - effect size matters more)
  Interpretation: Strong if means differ by > 5-10 years.


Step 3: Clean the text (feature engineering)
Clean and normalize the narrative text

In [326]:
def clean_text_negation(t):
    """
    Clean narrative text: lowercase, remove negated harm phrases,
    remove punctuation, and normalize whitespace.
    """
    if not isinstance(t, str):
        return ""
    t = t.lower()
    
    # Patterns for negated death/injury/harm
    # We remove the entire negated phrase to avoid leaving leftover context.
    patterns = [
        r'\bnot\s+death\b',
        r'\bno\s+death\b',
        r'\bdid\s+not\s+(result\s+in\s+)?death\b',
        r'\bdid\s+not\s+cause\s+death\b',
        r'\bwithout\s+death\b',
        r'\bnot\s+injury\b',
        r'\bno\s+injury\b',
        r'\bdid\s+not\s+(result\s+in\s+)?injury\b',
        r'\bdid\s+not\s+cause\s+injury\b',
        r'\bwithout\s+injury\b',
        r'\bnot\s+harm\b',
        r'\bno\s+harm\b',
        r'\bdid\s+not\s+cause\s+harm\b',
        r'\bwithout\s+harm\b'
    ]
    for pat in patterns:
        t = re.sub(pat, '', t)
    
    # Remove extra spaces (after removing phrases)
    t = re.sub(r'\s+', ' ', t)
    
    # Remove punctuation and special characters (keep letters and digits)
    t = re.sub(r'[^a-z0-9 ]', '', t)
    
    return t.strip()

In [327]:
# Replace old text_clean version with the negation-aware version
df['text_clean'] = df['text'].apply(clean_text_negation)

Step 5: Train/Test split 

In [328]:
# Define feature columns: text + device_class + manufacturer_d_name

X = df[[
    'text_clean', 
    'device_class', 
    'manufacturer_d_name', 
    'generic_name',              
    'patient_age_clean',         # Numerical
    'device_operator',           # Categorical
    'reporter_state_code',       # Categorical
    'medical_specialty',         # Categorical
    'patient_sex',               # Categorical
    'device_availability',       # Categorical
    'brand_name'
]] 

y = df['severity_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(f"Training size: {len(X_train):,}")
print(f"Test size: {len(X_test):,}")

Training size: 56,576
Test size: 18,859


Step 6: TF-IDF on training text ONLY
TF‑IDF vectorization
This converts text into weighted numerical features.

In [329]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import category_encoders as ce

# Define transformers
text_transformer = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9,
    stop_words='english',
    sublinear_tf=True
)

# Target encoder for device_class with smoothing
device_encoder = ce.TargetEncoder(
    cols=['device_class'],
    smoothing=10.0,
    handle_missing='value'
)

# Target encoder for manufacturer with smoothing
manufacturer_encoder = ce.TargetEncoder(
    cols=['manufacturer_d_name'],
    smoothing=10.0,              # Start with same smoothing; tune later
    handle_missing='value'
)

# --- New encoders
generic_encoder = ce.TargetEncoder(
    cols=['generic_name'],
    smoothing=20.0,
    handle_missing='value'
)

operator_encoder = ce.TargetEncoder(
    cols=['device_operator'],
    smoothing=10.0,
    handle_missing='value'
)

state_encoder = ce.TargetEncoder(
    cols=['reporter_state_code'],
    smoothing=10.0,
    handle_missing='value'
)

specialty_encoder = ce.TargetEncoder(
    cols=['medical_specialty'],
    smoothing=10.0,
    handle_missing='value'
)

sex_encoder = ce.TargetEncoder(
    cols=['patient_sex'],
    smoothing=10.0,
    handle_missing='value'
)

availability_encoder = ce.TargetEncoder(
    cols=['device_availability'],
    smoothing=10.0,
    handle_missing='value'
)

brand_encoder = ce.TargetEncoder(
    cols=['brand_name'],
    smoothing=10.0,
    handle_missing='value'
)

In [330]:
# --- Preprocessor definition ---

preprocessor = ColumnTransformer([
    ('text', text_transformer, 'text_clean'),
    ('device', device_encoder, ['device_class']),
    ('manufacturer', manufacturer_encoder, ['manufacturer_d_name']),
    ('generic', generic_encoder, ['generic_name']),
    ('operator', operator_encoder, ['device_operator']),
    ('state', state_encoder, ['reporter_state_code']),
    ('specialty', specialty_encoder, ['medical_specialty']),
    ('sex', sex_encoder, ['patient_sex']),
    ('availability', availability_encoder, ['device_availability']),
    # --- Simple imputer version of the model: This adds a binary column indicating whether age was missing.
    ('age', SimpleImputer(strategy='median', add_indicator=True), ['patient_age_clean']),
    ('brand', brand_encoder, ['brand_name'])
])

In [331]:
# Full pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(
        max_iter=4000,
        class_weight='balanced', # auto-computes weights
        solver='lbfgs',
        C=1.0,
        random_state=42
    ))
])

Step 7: Train the model
Train a Logistic Regression classifier

In [332]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('device', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Check Encoded Values for Interpretability

In [333]:
# Get the manufacturer encoder
manufacturer_enc = pipeline.named_steps['preprocessor'].named_transformers_['manufacturer']

# Get encoded values for top 10 manufacturers
top_manufacturers = df['manufacturer_d_name'].value_counts().head(10).index
unique_manufacturers = pd.DataFrame({'manufacturer_d_name': top_manufacturers})
encoded_values = manufacturer_enc.transform(unique_manufacturers)

print("Target encoded values for top manufacturers:")
for mfg, val in zip(unique_manufacturers['manufacturer_d_name'], encoded_values.values.flatten()):
    print(f"  {mfg[:50]}: {val:.4f}")

Target encoded values for top manufacturers:
  INSTITUT STRAUMANN AG: 1.0000
  CAREFUSION 303, INC.: 1.9996
  MEDTRONIC PUERTO RICO OPERATIONS CO.: 1.8179
  JJGC S.A.: 1.0000
  MEDTRONIC MINIMED: 1.9672
  DEXCOM, INC.: 1.9817
  CAREFUSION 303: 1.9951
  BOSTON SCIENTIFIC CORPORATION: 1.4114
  SYNTHES GMBH: 1.9891
  ALLERGAN (COSTA RICA): 1.0044


Insights:

Manufacturers like Straumann, JJGC, and Allergan have low encoded values → their reports are usually High severity (patient harm).

Manufacturers like Carefusion, Medtronic Minimed, and Synthes have high values → their reports are mostly Medium malfunctions.

This confirms that manufacturer is a strong predictor of severity – likely due to different product types and reporting cultures.

Evaluate the model

In [334]:
y_pred = pipeline.predict(X_test)

print("===== PERFORMANCE =====")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

===== PERFORMANCE =====
Accuracy: 92.18%

Classification Report:
              precision    recall  f1-score   support

    critical       0.20      0.77      0.32       219
        high       0.95      0.89      0.92      7379
      medium       0.96      0.95      0.95     11261

    accuracy                           0.92     18859
   macro avg       0.70      0.87      0.73     18859
weighted avg       0.95      0.92      0.93     18859


Confusion Matrix:
[[  168    38    13]
 [  400  6543   436]
 [  276   312 10673]]


Critical: 153 correct out of 188 → 81% recall. The model captures most critical events, but only 17% precision – meaning 83% of reports flagged as Critical are actually High or Medium. This is still the main weakness, but F1 improved from 0.25 to 0.28.

High: 9,676 correct out of 10,921 → 89% recall and 94% precision. Very reliable for High cases.

Medium: 9,648 correct out of 10,612 → 91% recall and 92% precision. Also solid.

Trade‑off: The model is now less overconfident – it doesn’t blindly predict High for everything. This is a healthier behavior for a triage system.

Threshold Tuning

Classification Report:
              precision    recall  f1-score   support

    critical       0.20      0.80      0.32       168

In [335]:
from sklearn.metrics import precision_recall_fscore_support

# Get probabilities for Critical class (index 0)
probs_critical = pipeline.predict_proba(X_test)[:, 0]
base_preds = pipeline.predict(X_test)

print("Threshold  Precision  Recall  F1")
for thresh in [0.40, 0.35, 0.30, 0.25, 0.20, 0.15]:
    adjusted_preds = base_preds.copy()
    # Override: if Critical probability >= threshold, force Critical (label 0)
    adjusted_preds[probs_critical >= thresh] = 0
    p, r, f, _ = precision_recall_fscore_support(y_test, adjusted_preds, labels=[0], average='weighted')
    print(f"{thresh:.2f}        {p:.3f}      {r:.3f}    {f:.3f}")

Threshold  Precision  Recall  F1
0.40        0.194      0.776    0.310
0.35        0.188      0.790    0.304
0.30        0.181      0.813    0.296
0.25        0.171      0.836    0.284
0.20        0.162      0.881    0.273
0.15        0.145      0.913    0.250


In [336]:
# Top predictive words (interpretability)
vectorizer = pipeline.named_steps['preprocessor'].named_transformers_['text']
feature_names = vectorizer.get_feature_names_out()
clf = pipeline.named_steps['clf']
coefs = clf.coef_

n_text_features = len(feature_names)
text_coefs = coefs[:, :n_text_features]

In [337]:
for idx, class_label in enumerate(le.classes_):
    top_indices = np.argsort(text_coefs[idx])[-20:][::-1]
    top_words = feature_names[top_indices]
    print(f"\nTop words for {class_label}:")
    for word in top_words:
        print(f"  {word}")


Top words for critical:
  death
  expired
  deceased
  passed away
  perforation
  patient deceased
  patient expired
  away
  patient passed
  lead insulation
  passed
  experienced cardiac
  died
  cause death
  cardiac
  syncope
  severe
  possible
  removed infection
  reported lead

Top words for high:
  explanted
  skin
  infection
  applicable
  bg
  revision
  osseointegrate
  unspecified
  lead capped
  estimated
  diabetes
  replaced patient
  additional adverse
  replaced
  capped
  implanted
  surgical
  xray
  capped replaced
  bg level

Top words for medium:
  reported
  medtronic employee
  employee caused
  employee
  impact
  involvement
  patient involvement
  use patient
  impedance
  impella
  lead implanted
  harm
  exhibited
  submitted manufacturers
  lead remains
  consequences
  patient harm
  intervention reported
  helix
  cable


Interpretability: View coefficients for device_class

The number tells the model: for this device class, the typical report is usually [High / Medium / Critical], so use that as a starting hint when the narrative text is unclear

In [338]:
# Get the target encoder from the pipeline
device_enc = pipeline.named_steps['preprocessor'].named_transformers_['device']

# Get the encoded values for each device class (fitted on training)
# We can transform a DataFrame of unique device classes
unique_classes = pd.DataFrame({'device_class': ['1','2','3','U','F','N']})
encoded_values = device_enc.transform(unique_classes)
print("Target encoded values per device class:")
for cls, val in zip(unique_classes['device_class'], encoded_values.values.flatten()):
    print(f"  {cls}: {val:.4f}")

Target encoded values per device class:
  1: 1.9564
  2: 1.2605
  3: 1.5462
  U: 1.8237
  F: 1.5854
  N: 1.9783


The numbers on the right (1.9454, 1.0393, etc.) are encoded Severity Scores that the model calculates for each device class. These scores live on a specific ruler:

0.0 = Critical (Death / Life‑threatening)

1.0 = High (Injury / Hospitalization)

2.0 = Medium (Malfunction / Product problem, no harm)